# Parameter layers: swapping a PCM

A parameter set is the union of independent layers: the operation, the coupling, the heat
transfer fluid, the PCM, the metal of the fins, and the geometry in two parts, the flow path
and the storage element (DESIGN D1). Each layer holds its own parameter names, so one part
of a set, such as the PCM, can be swapped without touching the rest. Each layer lists the
sources of its values in `"citations"`, and a composed set is a plain
`pybamm.ParameterValues`.

[Tutorial 4](getting_started/tutorial_4_parameter_values.ipynb) introduces parameter values and layers.

In [1]:
import numpy as np
import pybamm

import pytes
from pytes import parameter_layers as layers

for kind in layers.KINDS:
    print(f"{kind}: {', '.join(layers.available(kind))}")

operation: Finned tube discharge, Nallusamy2007, Raul2018
coupling: Finned tube default, Packed bed default
heat_transfer_fluid: Water, Hytherm 600
pcm: Paraffin (Nallusamy 2007), A164 (Raul 2018), PCM 10N, PCM 30, PCM 60
metal: Aluminium, Steel, Copper
finned_tube_flow_path: Finned tube default
finned_tube_storage_element: Finned tube default
packed_bed_flow_path: Nallusamy2007, Raul2018
packed_bed_storage_element: Nallusamy2007, Raul2018, Default capsule


Each family's default parameter values are a named composition. The finned-tube default
uses "PCM 60", a representative PCM for hot water melting at 60 °C.

In [2]:
print(layers.SETS["Finned tube default"])
parameter_values = layers.compose_set("Finned tube default")
print("citations:", parameter_values["citations"])
layers.get("pcm", "PCM 60")

{'operation': 'Finned tube discharge', 'coupling': 'Finned tube default', 'heat_transfer_fluid': 'Water', 'pcm': 'PCM 60', 'metal': 'Aluminium', 'finned_tube_flow_path': 'Finned tube default', 'finned_tube_storage_element': 'Finned tube default'}
citations: ['CRC']


{'citations': [],
 'PCM solid phase conductivity [W.m-1.K-1]': 0.6,
 'PCM liquid phase conductivity [W.m-1.K-1]': 0.5,
 'PCM solid phase density [kg.m-3]': 1400,
 'PCM liquid phase density [kg.m-3]': 1300,
 'PCM solid phase specific heat capacity [J.kg-1.K-1]': 2500,
 'PCM liquid phase specific heat capacity [J.kg-1.K-1]': 3000,
 'PCM latent heat [J.kg-1]': 250000.0,
 'PCM melting temperature [K]': 333.15,
 'PCM melting range [K]': 2.0,
 'PCM smoothing width [K]': 0.01}

## The same battery with another PCM

`swap` replaces a layer in place, and checks that the set already holds a layer of that
kind. Here the battery is the same, and so is the discharge from 70 °C with water at 10 °C:
only the PCM changes. The set's citations gain the sources of the new layer.

In [3]:
model = pytes.SharpFrontModel()
t = np.linspace(0, 3600, 121)
solutions = []
for pcm in ["PCM 60", "Paraffin (Nallusamy 2007)", "PCM 30"]:
    values = layers.compose_set("Finned tube default")
    layers.swap(values, pcm=pcm)
    capacity = pytes.energy_between(model, 273.15 + 10, 273.15 + 70, values)
    solution = pybamm.Simulation(model, parameter_values=values).solve(
        [0, t[-1]], t_interp=t
    )
    solutions.append(solution)
    print(
        f"{pcm}: {capacity / 3.6e6:.2f} kWh between 10 and 70 degC, citations {values['citations']}"
    )

pybamm.dynamic_plot(
    solutions,
    ["Outlet temperature [degC]", "Power [kW]", "Average liquid fraction"],
    labels=["PCM 60", "Paraffin (Nallusamy 2007)", "PCM 30"],
)

PCM 60: 2.45 kWh between 10 and 70 degC, citations ['CRC']


Paraffin (Nallusamy 2007): 1.26 kWh between 10 and 70 degC, citations ['CRC', 'Nallusamy2007']


PCM 30: 1.42 kWh between 10 and 70 degC, citations ['CRC']


interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…

The paraffin melts at 60 °C too, but stores less energy in the same volume: it is less
dense and has a lower latent heat. PCM 30 melts at 30 °C, so most of its heat comes out
below the 40 °C a hot-water supply needs.

## An operation that suits the PCM

An operation only makes sense relative to the PCM: a discharge starts with the PCM liquid
and cools it below its melting range. `operation.discharge` builds one from a PCM layer and
margins above its liquidus and below its solidus, with the mass flow rate always given.

In [4]:
operation = layers.operation.discharge("PCM 30", 5, 10, mass_flow_rate=0.02)
values = layers.compose_set("Finned tube default")
layers.swap(values, pcm="PCM 30", operation=operation)
operation

{'citations': [],
 'Reference temperature [K]': 291.65,
 'Initial temperature [K]': 309.65,
 'Inlet temperature [K]': 291.65,
 'Mass flow rate [kg.s-1]': 0.02}

## A layer of your own

`template` gives an empty layer of a kind, with every name it holds; `check` says what is
missing. A new layer can also start from a registered one.

In [5]:
template = layers.template("pcm")
try:
    layers.check("pcm", template)
except ValueError as error:
    print(error)

my_pcm = layers.get("pcm", "PCM 60")
my_pcm["PCM latent heat [J.kg-1]"] = 200e3
my_pcm["citations"] = []
layers.check("pcm", my_pcm)
values = layers.compose_set("Finned tube default")
layers.swap(values, pcm=my_pcm)
print("latent heat [J.kg-1]:", values["PCM latent heat [J.kg-1]"])

The pcm layer has no value for ['PCM latent heat [J.kg-1]', 'PCM liquid phase conductivity [W.m-1.K-1]', 'PCM liquid phase density [kg.m-3]', 'PCM liquid phase specific heat capacity [J.kg-1.K-1]', 'PCM melting range [K]', 'PCM melting temperature [K]', 'PCM smoothing width [K]', 'PCM solid phase conductivity [W.m-1.K-1]', 'PCM solid phase density [kg.m-3]', 'PCM solid phase specific heat capacity [J.kg-1.K-1]']
latent heat [J.kg-1]: 200000.0
